<a href="https://colab.research.google.com/github/routparam12/Python_qns/blob/main/Voice_analysis2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!ffmpeg -version | head

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --enable-

In [2]:
from google.colab import files
import os

uploaded = files.upload()

input_file = next(iter(uploaded))

print("Uploaded file:", input_file)
print("File size:", round(os.path.getsize(input_file) / 1024, 2), "KB")

Saving gehu_fungal_infection.wav to gehu_fungal_infection.wav
Uploaded file: gehu_fungal_infection.wav
File size: 130.08 KB


In [3]:
from IPython.display import Audio, display

display(Audio(input_file))

In [4]:
import subprocess
import json

def inspect_audio(file_path):

    command = [
        "ffprobe",
        "-v", "quiet",
        "-print_format", "json",
        "-show_format",
        "-show_streams",
        file_path
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=True
    )

    info = json.loads(result.stdout)

    audio_stream = next(
        stream for stream in info["streams"]
        if stream["codec_type"] == "audio"
    )

    print("FILE:", file_path)
    print("Codec:", audio_stream.get("codec_name"))
    print("Sample Rate:", audio_stream.get("sample_rate"), "Hz")
    print("Channels:", audio_stream.get("channels"))
    print("Channel Layout:", audio_stream.get("channel_layout"))
    print("Duration:", round(float(info["format"]["duration"]), 2), "seconds")

    return info

input_info = inspect_audio(input_file)

FILE: gehu_fungal_infection.wav
Codec: pcm_s16le
Sample Rate: 22050 Hz
Channels: 1
Channel Layout: None
Duration: 3.02 seconds


In [5]:
import subprocess

output_file = "normalized_audio.wav"

command = [
    "ffmpeg",
    "-y",
    "-i", input_file,
    "-ac", "1",
    "-ar", "16000",
    "-c:a", "pcm_s16le",
    output_file
]

subprocess.run(
    command,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=True
)

print("Created:", output_file)

Created: normalized_audio.wav


In [6]:
output_info = inspect_audio(output_file)

FILE: normalized_audio.wav
Codec: pcm_s16le
Sample Rate: 16000 Hz
Channels: 1
Channel Layout: None
Duration: 3.02 seconds


In [7]:
import os

input_size = os.path.getsize(input_file) / 1024
wav_size = os.path.getsize(output_file) / 1024

print(f"Original:       {input_size:.2f} KB")
print(f"Normalized WAV: {wav_size:.2f} KB")

Original:       130.08 KB
Normalized WAV: 94.44 KB


In [8]:
def normalize_audio(
    input_path,
    output_path="normalized_audio.wav",
    sample_rate=16000
):
    """
    Convert an audio file into:
    - WAV
    - PCM signed 16-bit
    - Mono
    - Selected sample rate
    """

    command = [
        "ffmpeg",
        "-y",
        "-i", input_path,
        "-ac", "1",
        "-ar", str(sample_rate),
        "-c:a", "pcm_s16le",
        output_path
    ]

    subprocess.run(
        command,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True
    )

    return output_path

In [13]:
import requests
print("Ready to use Sarvam AI REST API via requests!")

Ready to use Sarvam AI REST API via requests!


In [14]:
from google.colab import userdata

SARVAM_API_KEY = userdata.get("SARVAM_API_KEY")

print("API key loaded:", bool(SARVAM_API_KEY))

API key loaded: True


In [17]:
!pip install -U sarvamai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 478.6/478.6 kB 9.5 MB/s eta 0:00:00


In [18]:
from sarvamai import SarvamAI

client = SarvamAI(
    api_subscription_key=SARVAM_API_KEY
)

In [20]:
normalized_file = "normalized_audio.wav"

In [21]:
with open(normalized_file, "rb") as audio_file:

    response = client.speech_to_text.transcribe(
        file=audio_file,
        model="saaras:v4",
        mode="transcribe"
    )

print(response)

request_id='20261005_418ea8d6-10f6-429c-8a7b-f00f5c0d2f42' transcript='गेहूँ में फंगल इन्फेक्शन हो गया है।' timestamps=None language_code='hi-IN' language_probability=0.904


In [23]:
print(response.transcript)

गेहूँ में फंगल इन्फेक्शन हो गया है।


In [24]:
response = client.speech_to_text.transcribe(
    file=open("normalized_audio.wav", "rb"),
    model="saaras:v4",
    mode="transcribe"
)

print(response.transcript)

गेहूँ में फंगल इन्फेक्शन हो गया है।


In [25]:
!pip install -U sarvamai

In [26]:
import importlib.metadata

print(importlib.metadata.version("sarvamai"))

0.1.35


In [29]:
def transcribe_audio(
    audio_path,
    keyterms=None,
    language_code="hi-IN",
    mode="codemix"
):

    params = {
        "file": open(audio_path, "rb"),
        "model": "saaras:v4",
        "mode": mode,
        "language_code": language_code
    }

    if keyterms:
        params["keyterms"] = keyterms

    try:
        response = client.speech_to_text.transcribe(**params)
        return response.transcript

    finally:
        params["file"].close()

In [30]:
audio_path = "normalized_audio.wav"

baseline = transcribe_audio(
    audio_path,
    keyterms=None
)

print("BASELINE:")
print(baseline)

BASELINE:
गेहूँ में fungal infection हो गया है।


In [32]:
with_keyterms = transcribe_audio(
audio_path,
keyterms=agriculture_keyterms
)
print("WITH AGRICULTURE KEYTERMS:")
print(with_keyterms)

NameError: name 'agriculture_keyterms' is not defined